In [1]:
# Loading the required Python libraries
import pandas as pd
import numpy as np
import os

print("Python libraries loaded successfully.")

Python libraries loaded successfully.


In [2]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Importing the data files
import os

os.listdir("/content/drive/MyDrive/Colab Notebooks/NHANES")

['DPQ_J.XPT',
 'PAQ_J.xpt',
 'PAQY_J.xpt',
 'SLQ_J.xpt',
 'DEMO_J.xpt',
 'BMX_J.xpt',
 'WHQ_J.xpt',
 'DR1TOT_J.xpt',
 'Dataset_File_Summary.csv',
 'Preliminary_Missing_Data_Summary.csv']

In [20]:
# Reading each NHANES file
base_path = "/content/drive/MyDrive/Colab Notebooks/NHANES"

demo = pd.read_sas(f"{base_path}/DEMO_J.xpt", format="xport")
bmx = pd.read_sas(f"{base_path}/BMX_J.xpt", format="xport")
dpq = pd.read_sas(f"{base_path}/DPQ_J.XPT", format="xport")
paq = pd.read_sas(f"{base_path}/PAQ_J.xpt", format="xport")
slq = pd.read_sas(f"{base_path}/SLQ_J.xpt", format="xport")
diet = pd.read_sas(f"{base_path}/DR1TOT_J.xpt", format="xport")
whq = pd.read_sas(f"{base_path}/WHQ_J.xpt", format="xport")

print("All files loaded successfully.")

All files loaded successfully.


In [21]:
# Checking the number of participants and variables
datasets = {
    "Demographics": demo,
    "Body Measures": bmx,
    "Depression Screener": dpq,
    "Physical Activity": paq,
    "Sleep": slq,
    "Diet": diet,
    "Weight History": whq
}

summary = []

for name, dataframe in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": dataframe.shape[0],
        "Columns": dataframe.shape[1]
    })

dataset_summary = pd.DataFrame(summary)
dataset_summary

,Dataset,Rows,Columns
0,Demographics,9254,46
1,Body Measures,8704,21
2,Depression Screener,5533,11
3,Physical Activity,5856,17
4,Sleep,6161,11
5,Diet,8704,168
6,Weight History,6161,37


In [22]:
# Confirming the common participant identifier
for name, dataframe in datasets.items():
    print(name, "contains SEQN:", "SEQN" in dataframe.columns)

Demographics contains SEQN: True
Body Measures contains SEQN: True
Depression Screener contains SEQN: True
Physical Activity contains SEQN: True
Sleep contains SEQN: True
Diet contains SEQN: True
Weight History contains SEQN: True


In [23]:
# Merging the files

merged = demo.copy()

for dataframe in [bmx, dpq, paq, slq, diet, whq]:
    merged = merged.merge(
        dataframe,
        on="SEQN",
        how="left"
    )

print("Merged dataset shape:", merged.shape)
merged.head()

Merged dataset shape: (9254, 305)


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,WHD080U,WHD080L,WHQ225,WHD110,WHD120,WHD130,WHD140,WHQ150,WHQ190,WHQ200
0,93703.0,10.0,2.0,2.0,2.0,NaN,5.0,6.0,2.0,27.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,10.0,2.0,1.0,2.0,NaN,3.0,3.0,1.0,33.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,10.0,2.0,2.0,66.0,NaN,4.0,4.0,2.0,NaN,...,NaN,NaN,4.0,150.0,130.0,63.0,170.0,62.0,2.0,NaN
3,93706.0,10.0,2.0,1.0,18.0,NaN,5.0,6.0,2.0,222.0,...,NaN,NaN,5.0,NaN,NaN,NaN,150.0,17.0,2.0,NaN
4,93707.0,10.0,2.0,1.0,13.0,NaN,5.0,7.0,2.0,158.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Saving the merged raw dataset

merged.to_csv(
    "/content/NHANES_2017_2018_Merged_Raw.csv",
    index=False
)

print("Merged raw dataset saved.")

Merged raw dataset saved.


In [25]:
# Creating the adult study population (proposed study focuses on adults aged 20 and above)

adults = merged[merged["RIDAGEYR"] >= 20].copy()

print("All NHANES participants:", len(merged))
print("Participants aged 20 and above:", len(adults))

All NHANES participants: 9254
Participants aged 20 and above: 5569


In [26]:
# Excluding pregnant participants

if "RIDEXPRG" in adults.columns:
    adults = adults[
        (adults["RIDEXPRG"] != 1) |
        (adults["RIDEXPRG"].isna())
    ].copy()

print("Adult participants after pregnancy exclusion:", len(adults))

Adult participants after pregnancy exclusion: 5514


In [27]:
# Retaining participants with measured BMI

study_data = adults[
    adults["BMXBMI"].notna()
].copy()

print("Adults with valid measured BMI:", len(study_data))

Adults with valid measured BMI: 5120


In [28]:
# Checking for availability of depression-screening variables

phq_items = [
    "DPQ010", "DPQ020", "DPQ030",
    "DPQ040", "DPQ050", "DPQ060",
    "DPQ070", "DPQ080", "DPQ090"
]

for variable in phq_items:
    print(variable, variable in study_data.columns)

DPQ010 True
DPQ020 True
DPQ030 True
DPQ040 True
DPQ050 True
DPQ060 True
DPQ070 True
DPQ080 True
DPQ090 True


In [29]:
# Calculating the PHQ-9 psychological well-being score
# NHANES PHQ responses normally use values from 0 to 3.
# Codes such as 7 and 9 represent responses such as refusal or “do not know” and should not be treated as valid scores.

for variable in phq_items:

    # Correct SAS-imported zero values
    study_data.loc[
        (study_data[variable] > 0) &
        (study_data[variable] < 1),
        variable
    ] = 0

    # Convert refusal and don't-know responses to missing
    study_data.loc[
        study_data[variable].isin([7, 9]),
        variable
    ] = np.nan

study_data["PHQ9_TOTAL"] = study_data[phq_items].sum(
    axis=1,
    min_count=9
)

print(
    "Participants with complete PHQ-9:",
    study_data["PHQ9_TOTAL"].notna().sum()
)

study_data["PHQ9_TOTAL"].describe()


Participants with complete PHQ-9: 4729


,PHQ9_TOTAL
count,4729.000000
mean,3.223726
std,4.243321
min,0.000000
25%,0.000000
50%,2.000000
75%,5.000000
max,25.000000


In [30]:
#  Creating a preliminary healthy-weight target

def bmi_category(bmi):
    if pd.isna(bmi):
        return np.nan
    elif bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Healthy weight"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obesity"

study_data["BMI_CATEGORY"] = study_data["BMXBMI"].apply(bmi_category)

study_data["BMI_CATEGORY"].value_counts(dropna=False)

,count
BMI_CATEGORY,
Obesity,2142
Overweight,1650
Healthy weight,1246
Underweight,82


In [31]:
# Excluding underweight participants
#  1 = healthy weight
#  0 = above healthy weight

binary_data = study_data[
    study_data["BMXBMI"] >= 18.5
].copy()

binary_data["HEALTHY_WEIGHT"] = np.where(
    binary_data["BMXBMI"] < 25,
    1,
    0
)

binary_data["HEALTHY_WEIGHT"].value_counts()

,count
HEALTHY_WEIGHT,
0,3792
1,1246


In [32]:
#  Counting participants with BMI and PHQ-9 data

analysis_ready_initial = binary_data[
    binary_data["PHQ9_TOTAL"].notna()
].copy()

print(
    "Participants with valid BMI category and complete PHQ-9:",
    len(analysis_ready_initial)
)

print("\nHealthy-weight outcome distribution:")
print(
    analysis_ready_initial["HEALTHY_WEIGHT"]
    .value_counts()
)

print("\nOutcome percentages:")
print(
    analysis_ready_initial["HEALTHY_WEIGHT"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Participants with valid BMI category and complete PHQ-9: 4657

Healthy-weight outcome distribution:
HEALTHY_WEIGHT
0    3523
1    1134
Name: count, dtype: int64

Outcome percentages:
HEALTHY_WEIGHT
0    75.65
1    24.35
Name: proportion, dtype: float64


In [33]:
# Examining missing data

initial_variables = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "DMDMARTL",
    "INDFMPIR",
    "BMXBMI",
    "BMXWAIST",
    "PHQ9_TOTAL",
    "DR1TKCAL"
]

available_variables = [
    variable for variable in initial_variables
    if variable in analysis_ready_initial.columns
]

missing_summary = pd.DataFrame({
    "Variable": available_variables,
    "Missing_Count": [
        analysis_ready_initial[v].isna().sum()
        for v in available_variables
    ],
    "Missing_Percentage": [
        analysis_ready_initial[v].isna().mean() * 100
        for v in available_variables
    ]
})

missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Percentage"].round(2)
)

missing_summary

,Variable,Missing_Count,Missing_Percentage
0,RIDAGEYR,0,0.00
1,RIAGENDR,0,0.00
2,RIDRETH3,0,0.00
3,DMDEDUC2,0,0.00
4,DMDMARTL,0,0.00
5,INDFMPIR,573,12.30
6,BMXBMI,0,0.00
7,BMXWAIST,140,3.01
8,PHQ9_TOTAL,0,0.00
9,DR1TKCAL,257,5.52


In [34]:
#  Saving the preliminary analysis dataset

analysis_ready_initial.to_csv(
    "/content/NHANES_Preliminary_Analysis_Dataset.csv",
    index=False
)

print("Preliminary analysis dataset saved.")

Preliminary analysis dataset saved.


In [35]:
#  Saving the dataset summary tables

dataset_summary.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/NHANES/Dataset_File_Summary.csv",
    index=False
)

missing_summary.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/NHANES/Preliminary_Missing_Data_Summary.csv",
    index=False
)

**Sample Size Calculation**

In [36]:
# ============================================================
# SAMPLE SIZE CALCULATION FOR RESEARCH QUESTIONS RQ1-RQ4
# ============================================================

import math
import numpy as np
import pandas as pd

from scipy.stats import norm, f, ncf
from scipy.optimize import brentq
from statsmodels.stats.power import NormalIndPower


# ------------------------------------------------------------
# General assumptions
# ------------------------------------------------------------

alpha = 0.05
power = 0.80
confidence_level = 0.95
margin_of_error = 0.05


# ------------------------------------------------------------
# RQ1
# Association between depressive symptom severity and
# healthy weight status
#
# Planning approximation:
# Two-group comparison of healthy-weight proportions using
# a small standardized effect size, Cohen's h = 0.20.
# ------------------------------------------------------------

rq1_effect_size_h = 0.20

rq1_n_per_group = NormalIndPower().solve_power(
    effect_size=rq1_effect_size_h,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative="two-sided"
)

rq1_minimum_n = math.ceil(rq1_n_per_group * 2)


# ------------------------------------------------------------
# RQ2
# Incremental predictive value of depressive symptom severity
#
# Planning approximation:
# Test of incremental explained variation when PHQ-9 is added
# to a baseline model.
#
# Cohen's f-squared = 0.02 represents a small incremental effect.
# q = number of newly tested predictors.
# k = approximate total number of predictors in the expanded model.
# ------------------------------------------------------------

rq2_effect_size_f2 = 0.02
rq2_tested_predictors = 1
rq2_total_predictors = 10


def incremental_model_power(
    sample_size,
    effect_size_f2,
    tested_predictors,
    total_predictors,
    alpha_level
):
    """
    Approximate power for testing incremental model contribution
    using a noncentral F distribution.
    """

    df_numerator = tested_predictors
    df_denominator = sample_size - total_predictors - 1

    if df_denominator <= 0:
        return 0

    critical_f = f.ppf(
        1 - alpha_level,
        df_numerator,
        df_denominator
    )

    noncentrality_parameter = effect_size_f2 * sample_size

    calculated_power = ncf.sf(
        critical_f,
        df_numerator,
        df_denominator,
        noncentrality_parameter
    )

    return calculated_power


rq2_required_n = brentq(
    lambda n: incremental_model_power(
        sample_size=n,
        effect_size_f2=rq2_effect_size_f2,
        tested_predictors=rq2_tested_predictors,
        total_predictors=rq2_total_predictors,
        alpha_level=alpha
    ) - power,
    rq2_total_predictors + 2,
    10000
)

rq2_minimum_n = math.ceil(rq2_required_n)


# ------------------------------------------------------------
# RQ3
# Comparison of predictive model performance
#
# Confidence-interval calculation:
# n = Z^2 * p(1-p) / e^2
#
# p = 0.50 is used because it gives the most conservative,
# largest sample-size estimate.
# ------------------------------------------------------------

z_score = norm.ppf(1 - (1 - confidence_level) / 2)
expected_proportion = 0.50

rq3_minimum_n = math.ceil(
    (
        z_score ** 2
        * expected_proportion
        * (1 - expected_proportion)
    )
    / margin_of_error ** 2
)


# ------------------------------------------------------------
# RQ4
# Explainability and interpretation of the selected model
#
# The same confidence-interval criterion is applied because
# explanations will be generated from the validated analytical
# sample used for predictive modelling.
# ------------------------------------------------------------

rq4_minimum_n = rq3_minimum_n


# ------------------------------------------------------------
# Select the largest minimum sample required
# ------------------------------------------------------------

minimum_sample_sizes = [
    rq1_minimum_n,
    rq2_minimum_n,
    rq3_minimum_n,
    rq4_minimum_n
]

final_required_sample_size = max(minimum_sample_sizes)


# ------------------------------------------------------------
# Obtain the actual analytical sample size
# ------------------------------------------------------------

actual_sample_size = len(analysis_ready_initial)

sample_adequate = actual_sample_size >= final_required_sample_size


# ------------------------------------------------------------
# Create the synopsis sample-size table
# ------------------------------------------------------------

sample_size_table = pd.DataFrame({
    "Research Question": [
        "RQ1",
        "RQ2",
        "RQ3",
        "RQ4"
    ],

    "Method Used": [
        "Power analysis: two-proportion comparison",
        "Power analysis: incremental model contribution",
        "Confidence interval",
        "Confidence interval / validated-model sample"
    ],

    "Key Parameters": [
        (
            f"α = {alpha}, power = {power}, "
            f"Cohen's h = {rq1_effect_size_h}, "
            "two-sided test, equal group allocation"
        ),
        (
            f"α = {alpha}, power = {power}, "
            f"Cohen's f² = {rq2_effect_size_f2}, "
            f"{rq2_tested_predictors} tested predictor, "
            f"{rq2_total_predictors} total predictors"
        ),
        (
            f"95% CI, Z = {z_score:.3f}, "
            f"e = {margin_of_error}, p = {expected_proportion}"
        ),
        (
            f"95% CI, Z = {z_score:.3f}, "
            f"e = {margin_of_error}, p = {expected_proportion}"
        )
    ],

    "Minimum Sample Size (N)": [
        rq1_minimum_n,
        rq2_minimum_n,
        rq3_minimum_n,
        rq4_minimum_n
    ]
})


print("Sample Size Calculation")
print("-" * 70)

display(sample_size_table)

print(f"\nMinimum sample required across all research questions: "
      f"{final_required_sample_size}")

print(f"Actual analytical sample available: {actual_sample_size}")

print(f"Is the available sample adequate? {sample_adequate}")


# ------------------------------------------------------------
# Save the table for use in the synopsis and GitHub repository
# ------------------------------------------------------------

sample_size_table.to_csv(
    "/content/NHANES_Sample_Size_Calculation.csv",
    index=False
)

print(
    "\nSample-size table saved as "
    "'NHANES_Sample_Size_Calculation.csv'."
)

Sample Size Calculation
----------------------------------------------------------------------


,Research Question,Method Used,Key Parameters,Minimum Sample Size (N)
0,RQ1,Power analysis: two-proportion comparison,"α = 0.05, power = 0.8, Cohen's h = 0.2, two-si...",785
1,RQ2,Power analysis: incremental model contribution,"α = 0.05, power = 0.8, Cohen's f² = 0.02, 1 te...",395
2,RQ3,Confidence interval,"95% CI, Z = 1.960, e = 0.05, p = 0.5",385
3,RQ4,Confidence interval / validated-model sample,"95% CI, Z = 1.960, e = 0.05, p = 0.5",385



Minimum sample required across all research questions: 785
Actual analytical sample available: 4657
Is the available sample adequate? True

Sample-size table saved as 'NHANES_Sample_Size_Calculation.csv'.
